# Phase 1: Data Preprocessing

### Step 1: Automated Data Retrieval and Loading

In [1]:
import kagglehub
import shutil
import os
import pandas as pd

RAW_DATA_DIR = '../data/raw/'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

kaggle_cache_path = kagglehub.dataset_download("vincentgupo/dengue-cases-in-the-philippines")

for file_name in os.listdir(kaggle_cache_path):
    if file_name.endswith('.csv'):
        source_file = os.path.join(kaggle_cache_path, file_name)
        destination_file = os.path.join(RAW_DATA_DIR, file_name)
        shutil.copy(source_file, destination_file)

dengue_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'ph_dengue_cases2016-2020.csv'))
weather_df = pd.read_csv(os.path.join(RAW_DATA_DIR, 'phl-rainfall-subnat-full.csv'))

display(dengue_df.head())
display(weather_df.head())

c:\Personal Projects\dengue-predictor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.37k/6.37k [00:00<00:00, 8.06MB/s]

Extracting files...


,Month,Year,Region,Dengue_Cases,Dengue_Deaths
0,January,2016,Region I,705,1
1,February,2016,Region I,374,0
2,March,2016,Region I,276,0
3,April,2016,Region I,240,2
4,May,2016,Region I,243,1


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,1981-01-01,1,900566,PH01,426.0,4.906103,2.553130,NaN,NaN,NaN,NaN,131.15228,NaN,NaN,final
1,1981-01-11,1,900566,PH01,426.0,2.225352,1.920344,NaN,NaN,NaN,NaN,104.40741,NaN,NaN,final
2,1981-01-21,1,900566,PH01,426.0,1.985915,2.195070,9.117371,6.668545,NaN,NaN,97.09309,120.986560,NaN,final
3,1981-02-01,1,900566,PH01,426.0,1.246479,2.100157,5.457746,6.215571,NaN,NaN,87.97663,93.243110,NaN,final
4,1981-02-11,1,900566,PH01,426.0,0.875587,2.560642,4.107981,6.855868,NaN,NaN,77.71281,76.822556,NaN,final


### Step 2: Data Formatting and Corrections

In [2]:
dengue_df['Date'] = pd.to_datetime(dengue_df['Year'].astype(str) + '-' + dengue_df['Month'] + '-01', format='%Y-%B-%d')

weather_df = weather_df[weather_df['adm_level'] == 1].copy()
weather_df['date'] = pd.to_datetime(weather_df['date'])
weather_df = weather_df[(weather_df['date'].dt.year >= 2016) & (weather_df['date'].dt.year <= 2020)].copy()

weather_df['Year'] = weather_df['date'].dt.year
weather_df['Month'] = weather_df['date'].dt.month_name()

region_mapping = {
    'PH01': 'Region I', 'PH02': 'Region II', 'PH03': 'Region III',
    'PH04': 'Region IV-A', 'PH17': 'Region IV-B', 'PH05': 'Region V',
    'PH06': 'Region VI', 'PH07': 'Region VII', 'PH08': 'Region VIII',
    'PH09': 'Region IX', 'PH10': 'Region X', 'PH11': 'Region XI',
    'PH12': 'Region XII', 'PH13': 'NCR', 'PH14': 'CAR',
    'PH16': 'Region XIII', 'PH19': 'BARMM'
}
weather_df['Region'] = weather_df['PCODE'].map(region_mapping)

### Step 3: Aggregation and Merging

In [3]:
monthly_weather = weather_df.groupby(['Region', 'Year', 'Month']).agg({
    'rfh': 'sum',
    'rfh_avg': 'mean'
}).reset_index()

merged_df = pd.merge(dengue_df, monthly_weather, on=['Year', 'Month', 'Region'], how='inner')
merged_df = merged_df.sort_values(['Region', 'Date']).reset_index(drop=True)

### Step 4: Handling Null Values and Saving Output

In [4]:
merged_df['rfh'] = merged_df.groupby('Region')['rfh'].transform(lambda x: x.fillna(x.median()))
merged_df['rfh_avg'] = merged_df.groupby('Region')['rfh_avg'].transform(lambda x: x.fillna(x.median()))

os.makedirs('../data/processed/', exist_ok=True)
merged_df.to_csv('../data/processed/final_dengue_weather.csv', index=False)

display(merged_df.head())
print("Data Preprocessing Complete! Shape:", merged_df.shape)

,Month,Year,Region,Dengue_Cases,Dengue_Deaths,Date,rfh,rfh_avg
0,January,2016,BARMM,126,2,2016-01-01,21.523342,40.213978
1,February,2016,BARMM,92,2,2016-02-01,44.948402,38.875594
2,March,2016,BARMM,107,3,2016-03-01,35.624078,51.846384
3,April,2016,BARMM,109,4,2016-04-01,91.201476,53.762189
4,May,2016,BARMM,165,4,2016-05-01,241.466832,79.995030


Data Preprocessing Complete! Shape: (1020, 8)
